# 00 · Environment check

Run this notebook once after `uv sync` to confirm the machine can do the three things every later notebook depends on:

1. import TensorFlow and the project package,
2. compute audio features with the **same front end the Arduino firmware uses**,
3. convert a Keras model to a fully int8 TensorFlow Lite model and a C array.

No dataset is needed; the audio here is generated.

## 1. Versions

In [ ]:
import sys

import keras
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf

from fairy_kws import config
from fairy_kws.features import micro_features

print("Python    ", sys.version.split()[0])
print("TensorFlow", tf.__version__)
print("Keras     ", keras.__version__)
print("NumPy     ", np.__version__)
print("Devices   ", [d.device_type for d in tf.config.list_physical_devices()])
print("Labels    ", config.LABELS)

## 2. Audio features

A one-second sweep from 200 Hz to 4 kHz should show up as a rising diagonal in a 49 × 40 feature map (49 time slices, 40 frequency bins). That shape is what `micro_speech` feeds its model on the board.

In [ ]:
t = np.arange(config.CLIP_SAMPLES) / config.SAMPLE_RATE
sweep_hz = np.linspace(200, 4000, config.CLIP_SAMPLES)
audio = 0.5 * np.sin(2 * np.pi * np.cumsum(sweep_hz) / config.SAMPLE_RATE)

features = micro_features(tf.constant(audio, dtype=tf.float32)).numpy()
assert features.shape == (config.FEATURE_SLICES, config.FEATURE_BINS), features.shape
print("feature map:", features.shape, "range:", features.min(), "to", features.max())

fig, (ax_wave, ax_feat) = plt.subplots(2, 1, figsize=(8, 5), constrained_layout=True)
ax_wave.plot(t, audio, linewidth=0.5)
ax_wave.set(title="Waveform (generated sweep)", xlabel="seconds", xlim=(0, 1))
ax_feat.imshow(features.T, origin="lower", aspect="auto")
ax_feat.set(title="Micro front end features", xlabel="time slice", ylabel="frequency bin")
plt.show()

## 3. Model conversion

Build the `tiny_conv` architecture from `micro_speech` (untrained), convert it to a fully int8 TensorFlow Lite model, run it once, and write it out as a C array. This is the path a trained model will take to reach the board.

In [ ]:
num_labels = len(config.LABELS)
model = keras.Sequential(
    [
        keras.Input(shape=(config.FEATURE_SLICES, config.FEATURE_BINS, 1)),
        keras.layers.Conv2D(8, (10, 8), strides=(2, 2), padding="same", activation="relu"),
        keras.layers.Flatten(),
        keras.layers.Dense(num_labels, activation="softmax"),
    ],
    name="tiny_conv",
)
model.summary()

In [ ]:
rng = np.random.default_rng(0)


def representative_dataset():
    # Noise at different loudness levels, only so the converter can see a realistic
    # input range. Real training notebooks use real clips here.
    for _ in range(50):
        level = rng.uniform(0.01, 0.8)
        noise = rng.uniform(-level, level, config.CLIP_SAMPLES).astype(np.float32)
        yield [micro_features(tf.constant(noise)).numpy()[np.newaxis, ..., np.newaxis]]


converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_dataset
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.int8
converter.inference_output_type = tf.int8
tflite_model = converter.convert()

interpreter = tf.lite.Interpreter(model_content=tflite_model)
interpreter.allocate_tensors()
input_details = interpreter.get_input_details()[0]
output_details = interpreter.get_output_details()[0]
assert input_details["dtype"] == np.int8 and output_details["dtype"] == np.int8

scale, zero_point = input_details["quantization"]
quantized = np.clip(np.round(features / scale + zero_point), -128, 127).astype(np.int8)
interpreter.set_tensor(input_details["index"], quantized[np.newaxis, ..., np.newaxis])
interpreter.invoke()

print(f"model size      : {len(tflite_model) / 1024:.1f} kB")
print(
    f"input           : {input_details['shape']} int8, scale={scale:.4f}, zero_point={zero_point}"
)
print(f"output (int8)   : {interpreter.get_tensor(output_details['index'])[0]}")

In [ ]:
def to_c_array(model_bytes: bytes, name: str = "g_model") -> str:
    """Same result as `xxd -i`, without needing xxd installed."""
    rows = [
        "  " + ", ".join(f"0x{b:02x}" for b in model_bytes[i : i + 12])
        for i in range(0, len(model_bytes), 12)
    ]
    return (
        f"alignas(16) const unsigned char {name}[] = {{\n"
        + ",\n".join(rows)
        + f"\n}};\nconst int {name}_len = {len(model_bytes)};\n"
    )


config.MODELS_DIR.mkdir(exist_ok=True)
out_path = config.MODELS_DIR / "environment_check_model.cc"
out_path.write_text(to_c_array(tflite_model))
print("wrote", out_path.relative_to(config.REPO_ROOT), f"({out_path.stat().st_size} bytes)")
print("\nEnvironment OK.")